# Deep Dive: Prompt Chaining with Agents

## 📋 Summary

In **prompt chaining**, one LLM agent's output is the next agent's input. Each agent has **its own system prompt, temperature and output format**,
and a small function (the *chain*) passes the results along.

In the previous topic you chained *prompts*. Here we chain *agents*: named functions with a single responsibility, and we look at the
engineering details that make a chain work in practice:

| Detail | Why it matters |
|---|---|
| **Output contract** (fixed headings) | The next agent, or your code, can rely on the shape of what it receives |
| **Per-agent temperature** | Creative steps and analytical steps want different randomness |
| **Fan-in** | A later agent may need the outputs of *two* earlier agents |
| **Glue steps** | Sometimes you must transform one agent's output before the next agent can use it |
| **Observability** | Return every intermediate result so you can debug the chain |

### Key concepts in this notebook
| # | Concept | Complexity |
|---|---------|-----------|
| 1 | Two agents in a chain (researcher → writer) | 🟢 Beginner |
| 2 | Output contracts with fixed headings | 🟢 Beginner |
| 3 | Temperature per agent | 🟡 Intermediate |
| 4 | A longer chain with fan-in | 🟡 Intermediate |
| 5 | The glue step: free text vs. JSON | 🟡 Intermediate |
| 6 | Gates, retries and a traceable chain | 🔴 Advanced |
| 7 | Exercises | 🏋️ Practice |

**How to use this notebook:** run the cells in order (each section reuses helpers from the ones before). Cells marked `# TODO` are for you to fill in.
Running the notebook calls the OpenAI API and costs a few cents with the cheap model configured below.

## 0. Setup

In [ ]:
# Key input: in Colab, add OPENAI_API_KEY in the 🔑 Secrets panel (left sidebar).
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from getpass import getpass
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [ ]:
import difflib
import json
import re
import time

from openai import OpenAI
from IPython.display import Markdown, display

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-4.1-nano"  # cheap model, plenty for these experiments


def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.3) -> str:
    """Single-turn helper: returns the assistant text."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
    )
    return resp.choices[0].message.content


def call_llm_json(system_prompt: str, user_prompt: str) -> dict:
    """Like call_llm, but forces a JSON object back (the prompt must mention JSON)."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)


def show(label: str, text: str):
    display(Markdown(f"**{label}**\n\n{text}\n\n---"))

## 1. Two Agents in a Chain 🟢

The smallest chain: a **researcher** gathers structured notes, a **writer** turns them into an article.
Each agent is just a function with its own system prompt. The chain function calls them in order and hands over the result.

**Running example:** a short blog post about urban beekeeping.

In [ ]:
def researcher_agent(topic: str) -> str:
    system = """You are a research specialist who provides structured notes.
Keep every section under 80 words."""
    print(f"🔎 Researcher working on: {topic}")
    return call_llm(system, f"Research this topic: {topic}", temperature=0.7)


def writer_agent(topic: str, research: str) -> str:
    system = """You are a content writer who turns research notes into an engaging article.
Structure: short introduction, body, one-sentence conclusion. Maximum 150 words."""
    print(f"✍️  Writer working on: {topic}")
    return call_llm(system, f"Write an article about {topic} using ONLY these notes:\n\n{research}", temperature=0.3)


def run_simple_chain(topic: str) -> dict:
    research = researcher_agent(topic)
    article = writer_agent(topic, research)
    return {"research": research, "article": article}


out = run_simple_chain("urban beekeeping on city rooftops")
show("Research notes", out["research"])
show("Article", out["article"])

### 🧐 Observation
- Which parts of the article came directly from the research, and which did the writer invent? How does the instruction "using ONLY these notes" change that?
- The research came back in whatever shape the model liked. What if your code needed to pick out *one part* of it? That is the next section.

## 2. Output Contracts with Fixed Headings 🟢

Free-form output is fine for a human reader, but a chain is **code talking to code**. Tell the agent to always answer with the same headings
and you get an **output contract**: the next agent (or your parser) knows where to look.

We add a tiny parser that splits a heading-formatted answer into a dict.

In [ ]:
def structured_researcher(topic: str) -> str:
    system = """You are a research specialist. Always answer with EXACTLY these headings, each on its own line starting with '# ':
# OVERVIEW
# KEY POINTS
# RISKS
Keep every section under 60 words."""
    return call_llm(system, f"Research this topic: {topic}", temperature=0.2)


def split_sections(text: str) -> dict:
    """Split '# HEADING' style text into {HEADING: body}. Tolerates '##', '###' and bold markers."""
    sections, current = {}, None
    for line in text.splitlines():
        m = re.match(r"^#{1,3}\s*\**(.+?)\**\s*$", line.strip())
        if m:
            current = m.group(1).strip().upper()
            sections[current] = []
        elif current:
            sections[current].append(line)
    return {k: "\n".join(v).strip() for k, v in sections.items()}


notes = structured_researcher("urban beekeeping on city rooftops")
sections = split_sections(notes)
print("Headings found:", list(sections))
show("Only the RISKS section", sections.get("RISKS", "(missing)"))

### 🧐 Observation
- Now the writer could receive **only** `KEY POINTS`. When is passing less context better than passing everything?
- What happens if the model returns `**RISKS**` or `## Risks` instead of `# RISKS`? Why does the parser above tolerate that, and what would you do if a heading is missing entirely? (Section 6.)

## 3. Temperature per Agent 🟡

Temperature controls randomness. In a chain, **each agent can have its own**:

| Agent job | Temperature | Why |
|---|---|---|
| Extract, classify, analyze | 0 - 0.2 | You want the same answer every time |
| Brainstorm, research angles | 0.6 - 0.9 | Variety helps |
| Final polish / writing | 0.2 - 0.4 | Fluent but faithful to the input |

Let's measure it: call the same agent twice and compare how similar the two answers are.

In [ ]:
def similarity(a: str, b: str) -> float:
    return difflib.SequenceMatcher(None, a, b).ratio()


system = "You are a concise analyst."
prompt = "In two sentences, explain why rooftop farming is growing in big cities."

for temp in (0.0, 1.2):
    first = call_llm(system, prompt, temperature=temp)
    second = call_llm(system, prompt, temperature=temp)
    print(f"temperature={temp}: similarity between two runs = {similarity(first, second):.2f}")
    print("  run 1:", first[:120].replace("\n", " "), "...")
    print("  run 2:", second[:120].replace("\n", " "), "...")

### 🧐 Observation
- At temperature 0, are the two answers identical? (Even at 0 they are not *guaranteed* identical. Why might that be?)
- If a step feeds a **gate check** or a **parser**, which temperature would you give it?

## 4. A Longer Chain with Fan-in 🟡

Real chains have more than two steps, and a later step often needs **more than one earlier output** (*fan-in*).

**Running example:** a small **coffee roastery** deciding what to roast from a new batch of green beans.

```
Bean Analyst ──▶ Roast Planner ──▶ (glue) ──▶ Market Analyst ─┐
                       │                                        ▼
                       └──────────────────────────────▶ Production Strategist
```

The strategist receives **both** the roast plan and the market analysis.

In [ ]:
def bean_analyst_agent(bean: str) -> str:
    system = """You are a green-coffee quality analyst. Always use these headings:
# BEAN PROFILE
# ROAST POTENTIAL
# PROCESSING NOTES
Keep every section under 60 words."""
    print("☕ Bean Analyst working...")
    return call_llm(system, f"Analyze this green coffee: {bean}", temperature=0)


def roast_planner_agent(bean_report: str) -> str:
    system = """You are a roastery planner. Using the bean report, propose how to split the batch into roast levels.
Always use these headings:
# ROAST ALLOCATION (percentage per roast level, totalling 100%)
# ASSUMPTIONS
# NOTES FOR THE ROASTER
Keep every section under 60 words."""
    print("🔥 Roast Planner working...")
    return call_llm(system, f"Bean report:\n\n{bean_report}", temperature=0)


def market_analyst_agent(products_text: str) -> str:
    system = """You are a specialty-coffee market analyst. Always use these headings:
# DEMAND BY PRODUCT
# PRICE TRENDS
# RECOMMENDATIONS
Keep every section under 60 words."""
    print("📈 Market Analyst working...")
    return call_llm(system, f"Analyze the market for these products:\n{products_text}", temperature=0)


def production_strategist_agent(roast_plan: str, market_analysis: str) -> str:
    system = """You are a roastery production strategist. Balance what the roasters CAN do with what the market WANTS.
Always use these headings:
# PRODUCTION PLAN
# RATIONALE
# RISKS
Keep every section under 60 words."""
    print("🧠 Production Strategist working...")
    user = f"Roast plan:\n{roast_plan}\n\nMarket analysis:\n{market_analysis}"
    return call_llm(system, user, temperature=0)

## 5. The Glue Step: Free Text vs. JSON 🟡

The roast plan mentions the roast levels in prose, but the market analyst needs a **clean list of products**.
Something has to sit between them and turn one into the other: a **glue step**.

Option A: ask a model "list the products" and pass whatever text comes back.
Option B: ask for a **JSON object** and read the list in code. Let's compare.

In [ ]:
bean = "Ethiopia Yirgacheffe, washed, 1900 m altitude, 500 kg lot"
bean_report = bean_analyst_agent(bean)
roast_plan = roast_planner_agent(bean_report)

# Option A: free-text glue
products_free_text = call_llm("List all the roast-level products in the plan below, one per line.", roast_plan, temperature=0)
print("\n--- Option A (free text) ---")
print(products_free_text)

# Option B: JSON glue
products_json = call_llm_json(
    'Extract the roast-level products from the plan. Return JSON like {"products": ["Light roast", "Medium roast"]}.',
    roast_plan,
)
print("\n--- Option B (JSON) ---")
print(products_json)
print("Number of products:", len(products_json["products"]))

### 🧐 Observation
- In Option A the list may arrive with bullets, numbering or an intro sentence ("Here are the products:"). What could go wrong in the *next* agent, or in your code, if you try to count them?
- Option B costs the same but gives you a Python list. When is free text still the right glue?

## 6. Gates, Retries and a Traceable Chain 🔴

Chains fail in two ways: an API call **raises**, or an agent returns something **valid but wrong** (a missing heading, an empty section).
Two small tools cover most cases:

1. A **gate**: a cheap check between steps (`has_headings`), so bad output stops the chain early.
2. A **retry wrapper**: `run_step` reruns an agent when the gate fails or the call raises.

And the chain should **return every intermediate result** (not just the last one) so you can see where things went wrong.

In [ ]:
def has_headings(text: str, required: list) -> tuple:
    """Gate: are all required headings present? Returns (ok, missing)."""
    found = split_sections(text).keys()
    missing = [h for h in required if not any(f.startswith(h) for f in found)]
    return (not missing, missing)


def run_step(name: str, fn, *args, required_headings=None, retries: int = 2):
    """Run one agent; retry if it raises or fails the heading gate. Returns the agent output."""
    last_problem = ""
    for attempt in range(1, retries + 2):
        start = time.perf_counter()
        try:
            output = fn(*args)
        except Exception as e:  # network hiccup, rate limit...
            last_problem = f"raised {type(e).__name__}: {e}"
            print(f"   ⚠️  {name} attempt {attempt}: {last_problem}")
            continue
        if required_headings:
            ok, missing = has_headings(output, required_headings)
            if not ok:
                last_problem = f"missing headings {missing}"
                print(f"   ⚠️  {name} attempt {attempt}: {last_problem}")
                continue
        print(f"   ✅ {name} ok in {time.perf_counter() - start:.1f}s (attempt {attempt})")
        return output
    raise RuntimeError(f"{name} failed after {retries + 1} attempts ({last_problem})")


def run_roastery_chain(bean: str) -> dict:
    """Full chain. Returns ALL intermediate outputs so failures are easy to locate."""
    results = {"bean": bean}
    results["bean_report"] = run_step("bean_analyst", bean_analyst_agent, bean,
                                      required_headings=["BEAN PROFILE", "ROAST POTENTIAL"])
    results["roast_plan"] = run_step("roast_planner", roast_planner_agent, results["bean_report"],
                                     required_headings=["ROAST ALLOCATION"])
    # glue step: structured, so the next agent gets a clean list
    products = call_llm_json(
        'Extract the roast-level products from the plan. Return JSON like {"products": ["Light roast"]}.',
        results["roast_plan"],
    )["products"]
    results["products"] = products
    results["market"] = run_step("market_analyst", market_analyst_agent, "\n".join(f"- {p}" for p in products),
                                 required_headings=["DEMAND BY PRODUCT", "RECOMMENDATIONS"])
    results["plan"] = run_step("production_strategist", production_strategist_agent,
                               results["roast_plan"], results["market"],
                               required_headings=["PRODUCTION PLAN", "RISKS"])
    return results


chain_out = run_roastery_chain("Colombia Huila, honey process, 1700 m altitude, 300 kg lot")
show("Final production plan", chain_out["plan"])
print("Intermediate keys kept for debugging:", list(chain_out))

### 🧐 Observation
- If the final plan looks wrong, which keys of `chain_out` would you inspect first, and in what order?
- `run_step` retries with the *same* prompt. When would a retry with a **modified** prompt (adding the missing headings as feedback) work better?

## 7. Exercises 🏋️

### Exercise 1: Build a 3-agent chain in a new domain
**Domain:** planning a small community event.
1. `venue_analyst_agent(event_idea)`: proposes venue requirements (use fixed headings).
2. `budget_planner_agent(venue_report)`: turns the requirements into a rough budget.
3. `risk_reviewer_agent(venue_report, budget)`: **fan-in**, reviews both and lists the top risks.

Give each agent its own system prompt and a sensible temperature, and return **all** intermediate results in a dict.

In [ ]:
def venue_analyst_agent(event_idea: str) -> str:
    system = ""  # TODO: role + fixed headings + length limit
    # TODO: call call_llm(...) with a suitable temperature and return the text
    pass


def budget_planner_agent(venue_report: str) -> str:
    # TODO
    pass


def risk_reviewer_agent(venue_report: str, budget: str) -> str:
    # TODO: this agent uses TWO earlier outputs
    pass


def run_event_chain(event_idea: str) -> dict:
    # TODO: chain the three agents and return {"venue": ..., "budget": ..., "risks": ...}
    pass


# run_event_chain("A neighbourhood board-game night for 40 people")

### Exercise 2: A gate that gives feedback
`run_step` retries with the *same* prompt. Write `run_step_with_feedback` so that when the heading gate fails, the retry **tells the agent what was missing**
(for example by appending "Your previous answer was missing the headings: [...]" to the user prompt). Print each attempt.

Design question: which agent functions in this notebook would need a small change so that they can accept that extra feedback?

In [ ]:
def run_step_with_feedback(name: str, fn, first_input: str, required_headings: list, retries: int = 2) -> str:
    # TODO: call fn, run has_headings, and on failure call fn again with feedback included in the input
    pass

### Exercise 3: Trim the context
The strategist currently receives the **complete** roast plan and market analysis.
Use `split_sections` to send it only `ROAST ALLOCATION` and `RECOMMENDATIONS`, then compare the final plan and think about what you gained and lost.

In [ ]:
# TODO: extract the two sections from chain_out["roast_plan"] and chain_out["market"]
# TODO: call production_strategist_agent with the trimmed inputs and compare with chain_out["plan"]

## 🏁 Summary & Key Takeaways

| Concept | Rule of thumb |
|---|---|
| One agent, one job | A focused system prompt beats a long "do everything" prompt |
| Output contracts | Fixed headings (or JSON) make outputs parseable and steps replaceable |
| Temperature | Low for analysis/extraction/gates, moderate for ideation, low-moderate for writing |
| Fan-in | Later agents can combine several earlier outputs, so design each output to be reusable |
| Glue steps | Prefer **JSON** when code has to read the result |
| Robustness | Gate between steps, retry, and **return every intermediate result** |

### What's next?
- **Deep Dive 3: Routing**: instead of running every agent in a fixed line, let a router pick *which* agent handles each request.